In [ ]:
from transformers import pipeline

# ---------------------------------------------------------
# 1. Classify the chunk
# ---------------------------------------------------------

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

domain_labels = [
    "Data Governance",
    "Data Quality",
    "Master Data Management",
    "Data Engineering",
    "Artificial Intelligence",
    "Data Architecture",
    "Analytics"
]

content_type_labels = [
    "Concept",
    "Architecture",
    "Implementation",
    "Best Practice",
    "Problem",
    "Case Study",
    "Definition"
]



c:\Users\prahn\anaconda3\envs\lab_sbx\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\prahn\anaconda3\envs\lab_sbx\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prahn\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activa

NameError: name 'chunk' is not defined

In [3]:
# ---------------------------------------------------------
# Input chunk
# ---------------------------------------------------------

chunk = """
Collibra can harvest technical metadata and lineage from
Databricks and Power BI. The metadata can be used to connect
business glossary terms and critical data elements to physical
data assets. This provides a governed view of how data flows
from source systems through analytical platforms to reports.
"""


In [4]:

domain_result = classifier(
    chunk,
    candidate_labels=domain_labels,
    multi_label=True
)

content_type_result = classifier(
    chunk,
    candidate_labels=content_type_labels,
    multi_label=True
)


# ---------------------------------------------------------
# 2. Extract named entities
# ---------------------------------------------------------

ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

entities = ner(chunk)


# ---------------------------------------------------------
# 3. Build metadata
# ---------------------------------------------------------

metadata = {
    "domain": [
        {
            "label": label,
            "score": round(score, 3)
        }
        for label, score in zip(
            domain_result["labels"],
            domain_result["scores"]
        )
        if score >= 0.30
    ],

    "content_type": [
        {
            "label": label,
            "score": round(score, 3)
        }
        for label, score in zip(
            content_type_result["labels"],
            content_type_result["scores"]
        )
        if score >= 0.30
    ],

    "entities": [
        {
            "text": entity["word"],
            "type": entity["entity_group"],
            "score": round(entity["score"], 3)
        }
        for entity in entities
    ]
}


# ---------------------------------------------------------
# 4. Display
# ---------------------------------------------------------

from pprint import pprint

pprint(metadata)

c:\Users\prahn\anaconda3\envs\lab_sbx\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prahn\.cache\huggingface\hub\models--dslim--bert-base-NER. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 922.55it/s]
[transformers] BertForTokenClassification L

{'content_type': [{'label': 'Definition', 'score': 0.624},
                  {'label': 'Architecture', 'score': 0.541},
                  {'label': 'Implementation', 'score': 0.483},
                  {'label': 'Case Study', 'score': 0.466},
                  {'label': 'Best Practice', 'score': 0.457},
                  {'label': 'Concept', 'score': 0.449},
                  {'label': 'Problem', 'score': 0.332}],
 'domain': [{'label': 'Data Governance', 'score': 0.754},
            {'label': 'Data Quality', 'score': 0.631},
            {'label': 'Data Architecture', 'score': 0.617},
            {'label': 'Master Data Management', 'score': 0.583},
            {'label': 'Analytics', 'score': 0.532},
            {'label': 'Data Engineering', 'score': 0.334}],
 'entities': [{'score': np.float32(0.746), 'text': 'Colli', 'type': 'ORG'},
              {'score': np.float32(0.995), 'text': 'Databricks', 'type': 'ORG'},
              {'score': np.float32(0.965), 'text': 'Power BI', 'type': 'ORG'

In [5]:
# pip install sentence-transformers torch

from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np


# ---------------------------------------------------------
# 1. Sample chunk
# ---------------------------------------------------------

chunk = """
The organization uses Collibra as the enterprise data governance
platform. Business terms such as Customer and Product are maintained
in the business glossary and are linked to critical data elements.
Technical lineage is harvested from Databricks and Power BI so that
business users can understand where important data originates and
how it flows through downstream reports.
"""


# ---------------------------------------------------------
# 2. Sample controlled glossary
# ---------------------------------------------------------

glossary = [
    {
        "id": "G001",
        "term": "Business Glossary",
        "definition": (
            "A governed collection of business terms, definitions, "
            "and related metadata used to establish a common business vocabulary."
        ),
        "aliases": ["business terms", "business vocabulary"]
    },
    {
        "id": "G002",
        "term": "Critical Data Element",
        "definition": (
            "A data element that is considered critical to important "
            "business processes, reporting, risk, or regulatory requirements."
        ),
        "aliases": ["CDE", "critical data elements"]
    },
    {
        "id": "G003",
        "term": "Data Lineage",
        "definition": (
            "Information describing the origin, movement, transformation, "
            "and downstream flow of data between systems and data assets."
        ),
        "aliases": ["lineage", "technical lineage"]
    },
    {
        "id": "G004",
        "term": "Master Data Management",
        "definition": (
            "Processes and technology used to manage authoritative and "
            "consistent master data across an organization."
        ),
        "aliases": ["MDM", "master data"]
    },
    {
        "id": "G005",
        "term": "Data Quality",
        "definition": (
            "Processes and measurements used to assess and improve the "
            "accuracy, completeness, consistency, validity, and timeliness of data."
        ),
        "aliases": ["DQ", "data quality management"]
    },
    {
        "id": "G006",
        "term": "Data Product",
        "definition": (
            "A governed, reusable data asset designed to provide data "
            "to consumers for a defined business purpose."
        ),
        "aliases": ["data products"]
    }
]


# ---------------------------------------------------------
# 3. Load models
# ---------------------------------------------------------

# Bi-encoder:
# Used ONLY for inexpensive candidate generation.
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Cross-encoder:
# Used for contextual verification.
cross_encoder = CrossEncoder(
    "cross-encoder/nli-deberta-v3-base"
)


# ---------------------------------------------------------
# 4. Candidate generation
# ---------------------------------------------------------

# Create text representation of each glossary concept.
glossary_texts = [
    f"{item['term']}: {item['definition']}"
    for item in glossary
]

# Embed glossary concepts.
glossary_embeddings = embedding_model.encode(
    glossary_texts,
    normalize_embeddings=True
)

# Embed the chunk.
chunk_embedding = embedding_model.encode(
    chunk,
    normalize_embeddings=True
)

# Cosine similarity because vectors are normalized.
similarities = np.dot(
    glossary_embeddings,
    chunk_embedding
)

# Select top N candidates.
TOP_K = 4

candidate_indexes = np.argsort(similarities)[::-1][:TOP_K]

print("\n=== Candidate Generation ===")

candidates = []

for index in candidate_indexes:
    item = glossary[index]

    candidate = {
        "index": index,
        "term": item["term"],
        "definition": item["definition"],
        "embedding_score": float(similarities[index])
    }

    candidates.append(candidate)

    print(
        f"{item['term']:30s} "
        f"embedding_score={similarities[index]:.3f}"
    )


# ---------------------------------------------------------
# 5. Contextual verification using Cross-Encoder
# ---------------------------------------------------------

# We formulate the problem as NLI:
#
# Premise:
#     The actual chunk
#
# Hypothesis:
#     "This text discusses the concept of <term>."
#
# The cross-encoder evaluates the two pieces of text together.

pairs = []

for candidate in candidates:

    hypothesis = (
        f"This text discusses the concept of "
        f"{candidate['term']}. "
        f"The concept means: {candidate['definition']}"
    )

    pairs.append([chunk, hypothesis])


# Get NLI probabilities.
#
# apply_softmax=True gives probabilities for:
# contradiction / neutral / entailment
#
# We determine the exact label positions from the model itself.

nli_scores = cross_encoder.predict(
    pairs,
    apply_softmax=True
)

label_map = cross_encoder.model.config.id2label

print("\nModel labels:")
print(label_map)


# Find the entailment column dynamically.
entailment_index = None

for index, label in label_map.items():
    if "ENTAIL" in label.upper():
        entailment_index = int(index)
        break

if entailment_index is None:
    raise RuntimeError("Could not find entailment label")


# ---------------------------------------------------------
# 6. Produce metadata
# ---------------------------------------------------------

metadata_terms = []

print("\n=== Contextual Verification ===")

for candidate, scores in zip(candidates, nli_scores):

    entailment_score = float(scores[entailment_index])

    print(
        f"{candidate['term']:30s} "
        f"embedding={candidate['embedding_score']:.3f} "
        f"entailment={entailment_score:.3f}"
    )

    # IMPORTANT:
    # This threshold is only illustrative.
    # In a real system it should be determined using
    # a labelled evaluation set.

    if entailment_score >= 0.70:
        metadata_terms.append({
            "term": candidate["term"],
            "score": entailment_score,
            "method": "cross_encoder"
        })


# ---------------------------------------------------------
# 7. Final chunk metadata
# ---------------------------------------------------------

chunk_metadata = {
    "chunk_id": "chunk_001",
    "glossary_terms": metadata_terms
}


print("\n=== Final Metadata ===")
print(chunk_metadata)

c:\Users\prahn\anaconda3\envs\lab_sbx\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prahn\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3327.58it/s]
c:\Users\prahn\anaconda


=== Candidate Generation ===
Data Lineage                   embedding_score=0.637
Data Product                   embedding_score=0.570
Business Glossary              embedding_score=0.504
Critical Data Element          embedding_score=0.454

Model labels:
{0: 'contradiction', 1: 'entailment', 2: 'neutral'}

=== Contextual Verification ===
Data Lineage                   embedding=0.637 entailment=0.944
Data Product                   embedding=0.570 entailment=0.006
Business Glossary              embedding=0.504 entailment=0.158
Critical Data Element          embedding=0.454 entailment=0.001

=== Final Metadata ===
{'chunk_id': 'chunk_001', 'glossary_terms': [{'term': 'Data Lineage', 'score': 0.9444460868835449, 'method': 'cross_encoder'}]}
